# Multilingual: Laya's multilingual checkpoint with a head-only fine-tune

`laya:multilingual` reads many languages. Training only the decision head (the encoder stays frozen) is fast on a
laptop and keeps the encoder's languages intact (laya#320). The tickets below are short synthetic sentences in
Spanish, French and German; the held-out rows use phrasings that never appear in training.

Runs on: a laptop (CPU or MPS) or a GPU. First run downloads the multilingual checkpoint from Hugging Face.

Built on [Laya](https://github.com/NandhaKishorM/laya) (Apache-2.0) by Nandakishor M / Convai Innovations. decisionsmith is an independent project, not affiliated with TypeSafe AI or Convai Innovations.

In [1]:
# uv pip install "decisionsmith[laya]"
import random

import decisionsmith as ds

In [2]:
CORES = {
    "billing": [
        "me cobraron dos veces este mes", "la factura tiene un importe incorrecto", "quiero una copia de mi factura",
        "on m'a facturé deux fois", "ma facture indique un mauvais montant", "le paiement a échoué mais j'ai été débité",
        "mir wurde doppelt abgebucht", "meine Rechnung hat den falschen Betrag", "ich brauche eine Kopie der Rechnung",
        "el cargo de la tarjeta no es correcto", "je ne reconnais pas ces frais sur mon relevé",
        "die Zahlung ist fehlgeschlagen, aber das Geld ist weg",
    ],
    "technical": [
        "no puedo iniciar sesión desde la actualización", "la aplicación se cierra al abrir informes",
        "je n'arrive pas à me connecter", "l'export reste bloqué à 99 pour cent",
        "ich kann mich nicht anmelden", "die App stürzt beim Öffnen ab", "el panel muestra un error 500",
        "les e-mails de réinitialisation n'arrivent pas", "der Upload bricht nach einer Minute ab",
        "la sincronización dejó de funcionar", "la page reste blanche dans Safari", "die Suche liefert keine Ergebnisse",
    ],
    "sales": [
        "¿tienen descuentos para organizaciones sin fines de lucro?", "quiero mejorar mi plan",
        "pouvez-vous me faire un devis pour 50 postes ?", "existe-t-il une offre annuelle ?",
        "gibt es einen Rabatt für Schulen?", "ich möchte meinen Tarif upgraden", "¿hay un plan empresarial?",
        "quelle est la différence entre pro et team ?", "können wir über Mengenpreise sprechen?",
        "¿puedo pagar el plan anual con factura?", "y a-t-il un essai gratuit pour les équipes ?",
        "wen kann ich wegen einer Partnerschaft ansprechen?",
    ],
}


def make(cores, n, seed):
    rng = random.Random(seed)
    out = []
    for _ in range(n):
        team = rng.choice(sorted(cores))
        text = rng.choice(cores[team])
        out.append((text[0].upper() + text[1:], team))
    return out


# held-out rows use the last 3 phrasings of every team, which never appear in training
train_rows = make({t: xs[:-3] for t, xs in CORES.items()}, 300, seed=0)
test_rows = make({t: xs[-3:] for t, xs in CORES.items()}, 90, seed=1)
test_texts, truth = [t for t, _ in test_rows], [y for _, y in test_rows]
print(train_rows[:3])

[('¿hay un plan empresarial?', 'sales'), ('Ma facture indique un mauvais montant', 'billing'), ("Les e-mails de réinitialisation n'arrivent pas", 'technical')]


In [3]:
labels = ["billing", "technical", "sales"]
model = ds.model(labels, "laya:multilingual", question="Which team should handle this ticket?")


def accuracy(preds):
    return sum(p == y for p, y in zip(preds, truth)) / len(truth)


before = model.predict(test_texts)
print("zero-shot accuracy on held-out rows: %.2f" % accuracy(before))

<venv>/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


zero-shot accuracy on held-out rows: 0.71


In [4]:
report = model.train(train_rows, train="head", out="runs/multilingual")
print(report)

trained on 225 rows · accuracy 0.64 -> 1.00 on 45 held-out decisions · now using runs/multilingual
  note: only 45 held-out decisions, so these numbers are rough; more data helps
finetune: multilingual -> runs/multilingual

field  test  base_acc  accuracy  macro_f1  base_ece  ece  
-----  ----  --------  --------  --------  --------  -----
label  45    0.644     1.000     1.000     0.301     0.070
all    45    0.644     1.000     1.000     0.301     0.070

go: no
  - test split has 45 decisions; want 100 (add data)
saved: ./runs/multilingual


In [5]:
after = model.predict(test_texts)
print("held-out accuracy: %.2f zero-shot -> %.2f head-only fine-tune" % (accuracy(before), accuracy(after)))
for text in ["Me han cobrado dos veces la suscripción.", "Mon application plante au démarrage.",
             "Gibt es Rabatte für Vereine?"]:
    print("%-9s <- %s" % (model.predict(text), text))

held-out accuracy: 0.71 zero-shot -> 0.71 head-only fine-tune
billing   <- Me han cobrado dos veces la suscripción.
technical <- Mon application plante au démarrage.
sales     <- Gibt es Rabatte für Vereine?
